In [3]:
import requests
import csv
from bs4 import BeautifulSoup
import time
import random

# --- ⚙️ CONFIGURATION ⚙️ ---
TOURNAMENTS = [
    {
        "id": "4626469",
        "name": "PBL",
        "date": "09/06/2025"
    },
    {
        "id": "9460497",
        "name": "UBL 1-1",
        "date": "11/08/2025"
    },
    {
        "id": "608123",
        "name": "UBL 1-2",
        "date": "11/15/2025"
    },
    {
        "id": "8762799",
        "name": "UBL 1-3",
        "date": "11/23/2025"
    },
    {
        "id": "5669429",
        "name": "UBL 1-4",
        "date": "11/29/2025"
    },
    {
        "id": "7826390",
        "name": "UBL 1-5",
        "date": "11/30/2025"
    }
]

def scrape_tournament(tid, name, date, csv_filename):
    print(f"📊 Scraping Tournament: {name} ({date}) with ID: {tid}")
    # Placeholder for actual scraping logic per tournament
    # This function can be expanded to scrape specific tournament data if needed

    # --- Dynamic URLs ---
    BASE_URL = f"https://tcg.sfc-jpn.jp/loginnum.asp?tid={tid}&MMP=&flu=&Exclusive=0"
    LOGIN_ACTION_URL = "https://tcg.sfc-jpn.jp/login_bin.asp"
    PROFILE_URL = f"https://tcg.sfc-jpn.jp/tour.asp?tid={tid}&kno=1&znt=1&MMP=&flu=&Exclusive=0"
    
    # Create session for tournament and fetch list of players from dropdown
    dropdown_results = []
    try:
        session = requests.Session()
        resp = session.get(BASE_URL)
        resp.encoding = resp.apparent_encoding
        soup = BeautifulSoup(resp.text, 'html.parser')

        select = soup.find('select', attrs={'name': 'SelectShikibetsuNo'})
        if not select:
            print("❌ Error: Dropdown not found.")
            return

        for opt in select.find_all('option'):
            val = opt.get('value')
            text = opt.get_text(strip=True)
            if val and val.strip():
                dropdown_results.append({'id': val, 'dropdown_text': text})
    except Exception as e:
        print(f"❌ Connection Error: {e}")
        return

    players_in_tournament = []
    for i, p in enumerate(dropdown_results):
        current_id = p['id']
        dropdown_txt = p['dropdown_text']

        with requests.Session() as session:
            session.headers.update({'User-Agent': 'Mozilla/5.0', 'Referer': BASE_URL})
            try:
                # Login handshake
                session.get(BASE_URL)
                payload = {
                    'tid': tid,
                    'ShikibetsuNo': current_id,
                    'SelectShikibetsuNo': current_id,
                    'Dummy': '', 'MMP': '', 'flu': '', 'Exclusive': '0'
                }
                session.post(LOGIN_ACTION_URL, data=payload)

                # Fetch Name from Profile
                profile_resp = session.get(PROFILE_URL)
                profile_resp.encoding = profile_resp.apparent_encoding
                soup = BeautifulSoup(profile_resp.text, 'html.parser')

                found_name = None
                target_str = f"'{current_id}'"
                target_row = soup.find('tr', onclick=lambda x: x and target_str in x)

                if target_row:
                    cells = target_row.find_all('td')
                    if len(cells) >= 2:
                        found_name = cells[1].get_text(strip=True)

                if found_name:
                    print(f"✅ {dropdown_txt} {found_name}")
                    players_in_tournament.append([dropdown_txt, found_name, current_id, tid, name, date])

                time.sleep(random.uniform(0.6, 1.2))

            except Exception as e:
                print(f"⚠️ Error on ID {dropdown_txt}: {e}")
    
    # Save players_in_tournament to CSV (append mode)
    with open(csv_filename, 'a', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        if csvfile.tell() == 0:  # Write header if file is new
            writer.writerow(['Dropdown Text', 'Player Name', 'Player ID', 'Tournament ID', 'Tournament Name', 'Tournament Date'])
        writer.writerows(players_in_tournament)
    
    pass

def run_scraper():
    print("--- 🚀 Starting Optimized Scraper (Master List First) ---")
    
    # 1. Make new CSV file for this iteration
    timestamp = time.strftime("%Y%m%d-%H%M%S")
    csv_filename = f"tournament_players_{timestamp}.csv"
    
    # 2. Loop through each tournament
    for tournament in TOURNAMENTS:
        if tournament['name'] == "PBL" or tournament['name'] == "MBL":
            continue
        print(f"\n🎯 Processing Tournament: {tournament['name']} ({tournament['date']})")
        scrape_tournament(tournament['id'], tournament['name'], tournament['date'], csv_filename)

run_scraper()

--- 🚀 Starting Optimized Scraper (Master List First) ---

🎯 Processing Tournament: UBL 1-1 (11/08/2025)
📊 Scraping Tournament: UBL 1-1 (11/08/2025) with ID: 9460497
✅ ph00497299 Melvyn Germono
✅ ph00891794 Gideon Ong
✅ ph01304382 Marwin Adrias
✅ ph03198326 Mich
✅ ph03361636 Jeremie Pangan
✅ ph03540153 Jann Russell Ng
✅ ph03806355 Ellan Sanchez
✅ ph04510854 Lorenz Matthew Tech
✅ ph05914113 Raphael Yvan Tan
✅ ph07857878 ShawnYG
✅ ph07975647 Arthur Lance Armada
✅ ph08659166 Angelo Asonto
✅ ph08665642 Jay Phillip Bonillo
✅ ph09159382 Paolo Miguel Adriano
✅ ph09798881 Jhed Mark
✅ ph09824765 Kenneth Castillo
✅ ph10504706 Emmanuelle Eje
✅ ph10660363 Jhon Carlo Agpalza
✅ ph11796696 Laurence Leona
✅ ph13469953 Allan Kristoffer Agtutubo Velasquez
✅ ph14487390 JJPF
✅ ph14732383 Bryan Anthony Torre
✅ ph15249465 Irwin Cristan Soriano
✅ ph15920178 Kath Agbayani
✅ ph16206277 Rod
✅ ph16484152 Rafael George Filasol
✅ ph16557231 JB Mateo
✅ ph17873488 Yllaine Fealiz Lois Laggui Sabenecio
✅ ph18004658 Ray